# Round 4 (A100) — pay the detection debt, verify two laptop fixes on real data

**Self-contained. Connect an A100, then Runtime → Run all.** Each cell checks its
preconditions and prints a clear **STOP** rather than failing deep in a trace.

### What this round does, and why
Round 3 was laptop-only: it built the live LLM orchestration agent. Nothing about it has
ever touched real voxels, and two fixes made on the laptop are still unverified on real
data. This round is cheap (~1–2 h, no new downloads) and closes all three:

1. **Detection FROC re-confirm on subset0 — a DEBT.** Round 2's cell died in 0.24 s because
   the snapshot on Drive predated the `from eval...` sys.path fix. The fix is in the repo
   now; the preflight below proves it is in *this* snapshot before any A100 time is spent.
2. **Demo gallery, verifying the `propagation_drift` fix.** Round 2's gallery fired that
   flag on **18/22 lesions (82%)** because the repo still held a heuristic Round 1 had
   already rejected. The rewrite (z-extent > 2.5 × long axis) now lives in
   `src/oncoct/report/quality.py` — unit-tested, but never run on real masks. **Expect a
   low rate here. If it is still ~80%, the fix is wrong: report that, do not bury it.**
3. **First live agent run.** `scripts/run_agent.py` drives **Gemini** through the typed
   tools and writes the report *plus its tool-call trace*. The key is already filled in.
   This exact path was verified against the live Gemini API on the laptop, with the imaging
   faked, so the only untested part here is the imaging plane underneath it.

No LIDC download, no retraining, no CPU prep session — subset0 and the weights are already
in Drive from Round 2. Only the CONFIG cell may need editing.

In [ ]:
# 0. GPU present? ---------------------------------------------------------------------
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("STOP: no GPU. Runtime -> Change runtime type -> A100, then Run all.")
print(r.stdout.split("\n")[0])
print([l for l in r.stdout.splitlines() if any(k in l for k in ("A100", "L4", "Tesla", "MiB"))][:3])

In [ ]:
# 1. Mount Drive. ---------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.isdir("/content/drive/MyDrive"), "Drive did not mount - re-run this cell."
print("Drive mounted.")

## CONFIG — the only cell you may need to edit

Pre-filled with the paths Round 2 actually used (read off its executed notebook), so in the normal case this needs no edits at all. The preflight cell verifies every one of them.

In [ ]:
# 2. CONFIG ---------------------------------------------------------------------------
DRIVE          = "/content/drive/MyDrive"
PROJECT_DIR    = f"{DRIVE}/exposure_ai_Oncology/upload_to_drive_3.0"   # THIS uploaded folder

# --- LUNA16 data already in Drive from Rounds 1-2 ---
SUBSET0_DIR    = f"{DRIVE}/exposure_ai_Oncology/upload_to_drive/LUNA16/subset0"
ANNOTATIONS    = f"{DRIVE}/exposure_ai_Oncology/upload_to_drive/LUNA16/annotations.csv"
ANNOT_EXCLUDED = (f"{DRIVE}/exposure_ai_Oncology/upload_to_drive/LUNA16/"
                  "evaluationScript/evaluationScript/annotations/annotations_excluded.csv")

# --- caches / weights seeded by earlier rounds ---
CACHE_DIR       = f"{DRIVE}/oncoct/cache"
WEIGHTS_DIR     = f"{DRIVE}/oncoct/weights"
# Round 2's retrained head (n=1353). The gallery and the agent classify with THIS checkpoint.
MALIGNANCY_CKPT = f"{WEIGHTS_DIR}/malignancy_r2/malignancy_head.pt"

# --- knobs ---
RUN_DETECTION = True   # Job 1: the owed subset0 FROC re-confirm
N_DEMO_SCANS  = 8      # Job 2: same 8 scans as Round 2 -> flag rate is directly comparable
N_AGENT_SCANS = 1      # Job 3: free-tier is ~20 requests/DAY per model and one
                       # study is ~9, so 2 studies exhausts a model mid-run.

In [ ]:
# 3. PREFLIGHT — validate BEFORE spending A100 time. ----------------------------------
#    Round 2 burned a GPU session on a stale snapshot. This cell makes that impossible: it
#    proves the snapshot contains the specific fixes this round exists to exercise.
import os, glob
problems = []

REPO = os.path.join(PROJECT_DIR, "repo_snapshot")
if not os.path.isdir(REPO):
    problems.append(f"repo_snapshot missing under PROJECT_DIR: {PROJECT_DIR}")
if not os.path.isdir(SUBSET0_DIR):
    problems.append(f"SUBSET0_DIR missing: {SUBSET0_DIR}")
if RUN_DETECTION:
    for name, p in [("ANNOTATIONS", ANNOTATIONS), ("ANNOT_EXCLUDED", ANNOT_EXCLUDED)]:
        if not os.path.exists(p):
            problems.append(f"{name} missing: {p}")

def _read(rel):
    p = os.path.join(REPO, rel)
    return open(p, encoding="utf-8").read() if os.path.exists(p) else None

# The two laptop fixes this round is here to verify. If either is absent the snapshot is
# stale and the round would repeat Round 2's failure -- so refuse now, not 40 minutes in.
det = _read("scripts/run_detection_subset.py")
if det is None:
    problems.append("snapshot missing scripts/run_detection_subset.py")
elif "_REPO_ROOT" not in det or "sys.path.insert" not in det:
    problems.append("STALE SNAPSHOT: run_detection_subset.py lacks the repo-root sys.path fix "
                    "(commit 88c6002) - exactly what killed Round 2's detection cell.")

qual = _read("src/oncoct/report/quality.py")
if qual is None:
    problems.append("STALE SNAPSHOT: src/oncoct/report/quality.py missing (the drift-flag fix).")
elif "DRIFT_Z_EXTENT_RATIO" not in qual:
    problems.append("STALE SNAPSHOT: quality.py lacks the rewritten drift test.")

# Check POSITIVELY for markers only the implemented loop has. A "does it contain
# NotImplementedError" test false-positives: the finished orchestrator legitimately raises
# it for backends other than Anthropic, so that check would stop a perfectly good snapshot.
orch = _read("src/oncoct/agent/orchestrator.py")
if not orch or "stop_reason" not in orch:
    problems.append("STALE SNAPSHOT: orchestrator.py has no tool-use loop (no stop_reason "
                    "handling) - it is still the Round-2 stub; the agent job cannot run.")
tools_src = _read("src/oncoct/agent/tools.py")
if not tools_src or "assemble_report" not in tools_src:
    problems.append("STALE SNAPSHOT: agent/tools.py lacks assemble_report - the agent would "
                    "have no way to finalize a report.")
if not _read("scripts/run_agent.py"):
    problems.append("STALE SNAPSHOT: scripts/run_agent.py missing - no agent entry point.")
if not _read("src/oncoct/agent/gemini_client.py"):
    problems.append("STALE SNAPSHOT: gemini_client.py missing - the agent job needs it.")

if problems:
    raise SystemExit("STOP - fix before running:\n  - " + "\n  - ".join(problems))

sha = _read("SNAPSHOT_SHA.txt")
n_scans = len(glob.glob(os.path.join(SUBSET0_DIR, "*.mhd")))
print(f"OK. snapshot commit {sha.strip() if sha else '(unrecorded)'}")
print(f"    both laptop fixes present; subset0 has {n_scans} scans.")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

In [ ]:
# 4. Install deps + verify in a SUBPROCESS. -------------------------------------------
#    Colab preloads an old numpy in the KERNEL; installs upgrade it only ON DISK, so an
#    in-kernel `import monai` raises _blas_supports_fpe even though the stack is fine.
#    Every real step below runs via subprocess, so the kernel's numpy never matters.
import subprocess, sys, shutil

def sh(cmd, check=True):
    """Run a shell command and SHOW its output.

    Colab only captures Python-level stdout, not a subprocess's file descriptors, so a bare
    subprocess.run() sends everything to the kernel log where you never see it. In Round 4
    that silently swallowed the agent job's traceback: the notebook printed the command,
    printed nothing else, and the error was simply gone. Capture and re-print instead.
    """
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout[-4000:], flush=True)
    if r.stderr:
        print("--- stderr ---", flush=True)
        print(r.stderr[-4000:], flush=True)
    if r.returncode != 0:
        print(f"[exit {r.returncode}] {cmd}", flush=True)
        if check:
            raise SystemExit(f"STOP: command failed: {cmd}")
    return r

sh(f"cd '{REPO}' && pip -q install -e .")
sh("pip -q install monai nibabel SimpleITK matplotlib scikit-learn scikit-image pandas "
   "totalsegmentator")
sh("pip -q install -U 'numpy>=2.2,<3'")

chk = subprocess.run(
    [sys.executable, "-c",
     "import torch, monai, SimpleITK, numpy; "
     "print('numpy', numpy.__version__, '| torch', torch.__version__, "
     "'| cuda', torch.cuda.is_available())"],
    capture_output=True, text=True)
print(chk.stdout.strip())
if chk.returncode != 0:
    print(chk.stderr.strip()[-2000:])
    raise SystemExit("STOP: dependency import failed in a fresh process - read stderr above.")

os.environ["ONCOCT_BUNDLE_DIR"] = f"{WEIGHTS_DIR}/lung_nodule_ct_detection"
os.environ["TOTALSEG_HOME_DIR"] = f"{WEIGHTS_DIR}/totalsegmentator"
os.makedirs(os.environ["TOTALSEG_HOME_DIR"], exist_ok=True)
os.makedirs("/content/results/metrics", exist_ok=True)
os.makedirs("/content/results/weights", exist_ok=True)
if os.path.exists(MALIGNANCY_CKPT):
    shutil.copy(MALIGNANCY_CKPT, "/content/results/weights/malignancy_head.pt")
    print("Seeded Round-2 malignancy head (n=1353).")
else:
    print(f"NOTE: {MALIGNANCY_CKPT} not found - classification would use an untrained head.")

## JOB 1 — the owed subset0 FROC re-confirm

Round 1 measured **CPM 0.776 (95% CI 0.664–0.873)** out-of-fold on all 89 subset0 scans. Round 2 tried to re-confirm it and crashed on a stale snapshot. This should land near 0.776; a large deviation means something changed in the detection path — report it loudly rather than quietly adopting the new number.

In [ ]:
# 5. Detection + official FROC on ALL of subset0 (the honest, out-of-fold split). ------
import json
if RUN_DETECTION:
    sh(f"cd '{REPO}' && python scripts/run_detection_subset.py "
       f"--series-dir '{SUBSET0_DIR}' --out /content/results/metrics")
    sh(f"cd '{REPO}' && python eval/froc_luna16.py "
       f"--predictions /content/results/metrics/preds.csv "
       f"--annotations '{ANNOTATIONS}' --annotations-excluded '{ANNOT_EXCLUDED}' "
       f"--series-uids /content/results/metrics/evaluated_seriesuids.csv "
       f"--output /content/results/metrics")
    src = "/content/results/metrics/froc_subset.json"
    if os.path.exists(src):
        shutil.copy(src, "/content/results/metrics/froc_subset0.json")
        d = json.load(open(src))
        cpm = d.get("cpm")
        print(f"\n  subset0 (OUT-OF-FOLD) CPM = {cpm:.4f}   CI95 {d.get('ci95')}")
        print(f"  Round 1 reported 0.7755.  Delta = {cpm - 0.7755:+.4f}")
        print("  DEBT PAID." if abs(cpm - 0.7755) < 0.05
              else "  *** DEVIATES from Round 1 by >0.05 - investigate before quoting. ***")
    else:
        print("PROBLEM: FROC json not written; read the eval output above.")
else:
    print("RUN_DETECTION=False -> skipping the re-confirm (the debt stays unpaid).")

## JOB 2 — demo gallery, and the `propagation_drift` verdict

Same 8 subset0 scans as Round 2, so the flag rate is directly comparable. **Round 2: 18/22 lesions flagged (82%) using the heuristic Round 1 had already rejected.**

In [ ]:
# 6. MedSAM2 in an isolated venv, then the full pipeline on N scans. -------------------
MED = "/content/MedSAM2"
if not os.path.isdir(MED):
    sh("pip -q install virtualenv")
    sh(f"git clone https://github.com/bowang-lab/MedSAM2.git {MED}")
    sh("virtualenv /content/venv_medsam2")
    sh("/content/venv_medsam2/bin/pip -q install torch==2.5.1 torchvision==0.20.1 "
       "--index-url https://download.pytorch.org/whl/cu124")
    sh(f"cd {MED} && /content/venv_medsam2/bin/pip -q install -e '.[dev]' && bash download.sh",
       check=False)
os.environ["ONCOCT_MEDSAM2_PYTHON"] = "/content/venv_medsam2/bin/python"
os.environ["ONCOCT_MEDSAM2_ROOT"] = MED

scans = sorted(glob.glob(os.path.join(SUBSET0_DIR, "*.mhd")))[:N_DEMO_SCANS]
for s in scans:
    sh(f"cd '{REPO}' && python scripts/run_pipeline.py --series '{s}' "
       f"--out /content/results --cache '{CACHE_DIR}'", check=False)
print(f"Demo pipeline ran on {len(scans)} scans.")

In [ ]:
# 7. THE VERDICT on the drift-flag fix. -----------------------------------------------
import collections
flags, n_les = collections.Counter(), 0
for f in sorted(glob.glob("/content/results/reports/*.json")):
    for finding in json.load(open(f))["findings"]:
        n_les += 1
        for q in finding.get("quality_flags") or []:
            flags[q] += 1

drift = flags.get("propagation_drift", 0)
rate = drift / n_les if n_les else 0.0
print(f"lesions: {n_les}   flags: {dict(flags)}")
print(f"\n  propagation_drift: {drift}/{n_les} = {rate:.0%}")
print("  Round 2 (rejected heuristic): 18/22 = 82%")
if n_les == 0:
    print("  INCONCLUSIVE - no lesions produced; the gallery did not run.")
elif rate <= 0.25:
    print("  => FIX CONFIRMED on real masks. The flag discriminates again.")
else:
    print("  => FIX DID NOT WORK. Report this honestly; do NOT quote the flag as meaningful.")
print("\n  (outside_lung_parenchyma firing is EXPECTED at the 0.02 threshold - not a bug.)")

## JOB 3 — first live run of the orchestration agent (Gemini)

The key is already filled in below, so this just runs. The agent loop is provider-agnostic — `src/oncoct/agent/gemini_client.py` adapts Gemini's function calling to the same interface, and this exact path was verified against the live API on the laptop before this notebook was built.

In [ ]:
# 8. Live agent: Gemini drives the typed tools; the trace is written as evidence. ------
#    The loop is provider-agnostic (see src/oncoct/agent/gemini_client.py); only the client
#    differs. configs/pipeline.yaml already selects report.llm: gemini.
GEMINI_API_KEY = ""

if not GEMINI_API_KEY:
    try:                                    # fall back to a Colab secret if one is set
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY or ""
have_key = bool(GEMINI_API_KEY)
if not have_key:
    print("No GEMINI_API_KEY - SKIPPING the agent job. Jobs 1 and 2 still stand.")
else:
    print(f"Gemini key loaded ({len(GEMINI_API_KEY)} chars). Model: gemini-flash-latest")

if have_key:
    for s in scans[:N_AGENT_SCANS]:
        sh(f"cd '{REPO}' && python scripts/run_agent.py --series '{s}' "
           f"--out /content/results --cache '{CACHE_DIR}'", check=False)

    traces = sorted(glob.glob("/content/results/agent/*_trace.json"))
    reports = sorted(glob.glob("/content/results/agent/*_report.json"))
    print(f"\n  agent runs: {len(reports)} report(s), {len(traces)} trace(s)")
    for t in traces:
        d = json.load(open(t))
        tools = [c["tool"] for c in d["calls"]]
        print(f"   {d['study'][:28]}...  {d['n_tool_calls']} calls: {' -> '.join(tools)}")
    for rp in reports:
        d = json.load(open(rp))
        print(f"\n   impression: {d['impression']}")
        print(f"   findings: {len(d['findings'])}")
    for t in traces:                      # the trace now records WHY it stopped
        d = json.load(open(t))
        if d.get("status") == "failed" and d.get("failure"):
            print(f"   FAILED: {d['failure']['type']}: {d['failure']['message'][:300]}")
    if not reports:
        print("   No report. The trace above says why. Free-tier quota is the usual cause:")
        print("   ~20 requests/day PER MODEL; the client rolls over to another model, but if")
        print("   they are all spent you must wait for the daily reset or use a paid key.")

In [ ]:
# 9. Copy results to Drive + summary. -------------------------------------------------
dest = os.path.join(PROJECT_DIR, "results")
if os.path.isdir(dest):
    shutil.rmtree(dest)
shutil.copytree("/content/results", dest)
print("results/ ->", dest)
for root, _, fs in os.walk(dest):
    for f in fs:
        print("  ", os.path.relpath(os.path.join(root, f), dest))

print()
print("DONE. Download results/ and merge it back into the repo.")
print("Tell it, one line each:")
print("  1. subset0 CPM (JOB 1) - and whether the debt is paid")
print("  2. the propagation_drift rate (JOB 2) - fix confirmed or not")
print("  3. whether the agent produced a report (JOB 3), or what it failed on")
print("Disconnect the runtime now if it does not auto-stop (A100 bills per wall-clock).")